In [13]:
# Install the package (uncomment if not already installed)
#!pip install git+https://github.com/sebastianprobst/resonator_tools.git

In [14]:
import numpy as np
import pandas as pd
from resonator_tools import circuit

In [15]:
# List of file base names (without extension) to process
file_names = [
    'resontors_S21_4.80-4.90GHz_0.0dBm_20260716_104636',
    'resontors_TWPA_off_S21_4.80-4.90GHz_-10.0dBm_20260716_141933',
    'resontors_TWPA_off_S21_4.82-4.86GHz_-10.0dBm_20260716_143442',
    'resontors_TWPA_off_S21_6.64-6.67GHz_-10.0dBm_20260716_143719',
]

data_folder = 'Data2'

In [16]:
def fit_q_factors(csv_path):
    """Load an S21 sweep CSV and return the notch-port fit results dict."""
    df = pd.read_csv(csv_path)

    # Convert units
    freq = df['Frequency (GHz)'].values * 1e9
    mag_lin = 10**(df['Magnitude (dB)'].values / 20)
    phase_rad = np.deg2rad(df['Phase (deg)'].values)
    s21_complex = mag_lin * np.exp(1j * phase_rad)

    fit = circuit.notch_port()
    fit.add_data(freq, s21_complex)

    delay, a, alpha, fr, Ql, A2, frcal = fit.do_calibration(fit.f_data, fit.z_data_raw)
    fit.z_data = fit.do_normalization(fit.f_data, fit.z_data_raw, delay, a, alpha, A2, frcal)
    fit.fitresults = fit.circlefit(fit.f_data, fit.z_data, fr, Ql)

    return fit.fitresults

In [17]:
results = []

for name in file_names:
    csv_path = f'{data_folder}/{name}.csv'
    try:
        fitresults = fit_q_factors(csv_path)
        results.append({
            'file': name,
            'fr_Hz': fitresults['fr'],
            'Ql': fitresults['Ql'],
            'Qc_dia_corr': fitresults['Qc_dia_corr'],
            'Qi_dia_corr': fitresults['Qi_dia_corr'],
        })
        print(f"{name}\n"
              f"  fr = {fitresults['fr']/1e9:.4f} GHz | Ql = {fitresults['Ql']:.2e} | "
              f"Qc = {fitresults['Qc_dia_corr']:.2e} | Qi = {fitresults['Qi_dia_corr']:.2e}\n")
    except Exception as e:
        print(f"Failed to fit {name}: {e}\n")
        results.append({
            'file': name,
            'fr_Hz': np.nan,
            'Ql': np.nan,
            'Qc_dia_corr': np.nan,
            'Qi_dia_corr': np.nan,
        })

c:\Users\Benji\Resonator-Frequency-Simulation\.venv\Lib\site-packages\resonator_tools\circlefit.py:379: RuntimeWarning: The iteration is not making good progress, as measured by the 
 improvement from the last ten iterations.
  x0 = spopt.fsolve(func, 0.0, fprime=d_func)


resontors_S21_4.80-4.90GHz_0.0dBm_20260716_104636
  fr = 4.8415 GHz | Ql = 4.56e+02 | Qc = 8.38e+02 | Qi = 1.00e+03

resontors_TWPA_off_S21_4.80-4.90GHz_-10.0dBm_20260716_141933
  fr = 4.8425 GHz | Ql = 3.67e+03 | Qc = 4.81e+03 | Qi = 1.55e+04

resontors_TWPA_off_S21_4.82-4.86GHz_-10.0dBm_20260716_143442
  fr = 4.8423 GHz | Ql = 2.29e+03 | Qc = 2.84e+03 | Qi = 1.18e+04

resontors_TWPA_off_S21_6.64-6.67GHz_-10.0dBm_20260716_143719
  fr = 6.6554 GHz | Ql = 1.27e+03 | Qc = 2.74e+03 | Qi = 2.37e+03



In [18]:
results_df = pd.DataFrame(results)
results_df.to_csv(f'{data_folder}/Q_factor_results.csv', index=False)
results_df

,file,fr_Hz,Ql,Qc_dia_corr,Qi_dia_corr
0,resontors_S21_4.80-4.90GHz_0.0dBm_20260716_104636,4.841501e+09,455.980734,838.089648,1000.114675
1,resontors_TWPA_off_S21_4.80-4.90GHz_-10.0dBm_2...,4.842541e+09,3671.310182,4812.239778,15484.938742
2,resontors_TWPA_off_S21_4.82-4.86GHz_-10.0dBm_2...,4.842283e+09,2289.370833,2841.468323,11782.655812
3,resontors_TWPA_off_S21_6.64-6.67GHz_-10.0dBm_2...,6.655426e+09,1271.800881,2738.479629,2374.617353
